In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
data = pd.read_csv('wine_data.csv')

In [17]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


What is the most frequently occurring wine quality? What is the highest number in and the lowest number in the quantity column?


In [18]:
data.groupby(data['quality']).count()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
quality,,,,,,,,,,,
3,10,10,10,10,10,10,10,10,10,10,10
4,53,53,53,53,53,53,53,53,53,53,53
5,681,681,681,681,681,681,681,681,681,681,681
6,638,638,638,638,638,638,638,638,638,638,638
7,199,199,199,199,199,199,199,199,199,199,199
8,18,18,18,18,18,18,18,18,18,18,18


 from the above code chunk its clearly can be seen that quality 5 has the most frequency and highest number is 8 and the lowest is 3

In [19]:
most_frequent = data['quality'].mode()[0]
print("Most frequent quality:", most_frequent)

data['quality'].value_counts()
min = data['quality'].min()
max = data['quality'].max()
print("Min quality:", min)
print("Max quality:", max)



Most frequent quality: 5
Min quality: 3
Max quality: 8


How is `fixed acidity` correlated to the quality of the wine? How does the alcohol content affect the quality? How is the `free Sulphur dioxide` content correlated to the quality of the wine?



In [20]:
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

features = ['fixed acidity', 'alcohol', 'free sulfur dioxide']
X = data[features]
y = data['quality']

print("--- Pearson Correlation ---")
print(data[features + ['quality']].corr()['quality'])

# Scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_scaled = sm.add_constant(X_scaled) #In the linear regression model $Y = \beta_0 + \beta_1 X_1 + \dots + \beta_k X_k + \epsilon$, the term $\beta_0$ is the intercept (or constant).

model = sm.OLS(y, X_scaled).fit()

print("\n--- Regression Results ---")
print(model.summary().tables[1])



--- Pearson Correlation ---
fixed acidity          0.124052
alcohol                0.476166
free sulfur dioxide   -0.050656
quality                1.000000
Name: quality, dtype: float64

--- Regression Results ---
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   5.6360      0.018    322.030      0.000       5.602       5.670
fixed acidity           0.1252      0.018      7.050      0.000       0.090       0.160
alcohol                 0.3925      0.018     22.314      0.000       0.358       0.427
free sulfur dioxide     0.0056      0.018      0.316      0.752      -0.029       0.040


What is the average `residual sugar` for the best quality wine and the lowest quality wine in the dataset?


In [21]:
data.groupby(data['quality']).mean()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
quality,,,,,,,,,,,
3,8.360000,0.884500,0.171000,2.635000,0.122500,11.000000,24.900000,0.997464,3.398000,0.570000,9.955000
4,7.779245,0.693962,0.174151,2.694340,0.090679,12.264151,36.245283,0.996542,3.381509,0.596415,10.265094
5,8.167254,0.577041,0.243686,2.528855,0.092736,16.983847,56.513950,0.997104,3.304949,0.620969,9.899706
6,8.347179,0.497484,0.273824,2.477194,0.084956,15.711599,40.869906,0.996615,3.318072,0.675329,10.629519
7,8.872362,0.403920,0.375176,2.720603,0.076588,14.045226,35.020101,0.996104,3.290754,0.741256,11.465913
8,8.566667,0.423333,0.391111,2.577778,0.068444,13.277778,33.444444,0.995212,3.267222,0.767778,12.094444


In [22]:
data.groupby('quality')['residual sugar'].mean()

,residual sugar
quality,
3,2.635000
4,2.694340
5,2.528855
6,2.477194
7,2.720603
8,2.577778


In [23]:
quality_means = data.groupby('quality')['residual sugar'].mean()

lowest_avg = quality_means.iloc[0]
best_avg = quality_means.iloc[-1]

print(f"Lowest Quality Avg Sugar: {lowest_avg}")
print(f"Best Quality Avg Sugar: {best_avg}")

Lowest Quality Avg Sugar: 2.6350000000000002
Best Quality Avg Sugar: 2.5777777777777775


Does `volatile acidity` has an effect over the quality of the wine samples in the dataset?


In [25]:

# A negative value indicates that as acidity increases, quality tends to decrease.
correlation = data['volatile acidity'].corr(data['quality'])
print(f"Correlation between Volatile Acidity and Quality: {correlation:.4f}")

# Viewing the Mean Volatile Acidity for each Quality level
mean_acidity = data.groupby('quality')['volatile acidity'].mean()
print("\nMean Volatile Acidity per Quality Level:")
print(mean_acidity)



Correlation between Volatile Acidity and Quality: -0.3906

Mean Volatile Acidity per Quality Level:
quality
3    0.884500
4    0.693962
5    0.577041
6    0.497484
7    0.403920
8    0.423333
Name: volatile acidity, dtype: float64


Train a Decision Tree model and Random Forest Model separately to predict the Quality of the given samples of wine. Compare the Accuracy scores for both models.



Decision Tree


In [26]:
features = ['fixed acidity', 'alcohol', 'free sulfur dioxide']
X = data[features]
y = data['quality']
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)



In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
classifer = DecisionTreeClassifier(criterion='entropy', random_state=42)
classifer.fit(x_train, y_train)

DecisionTreeClassifier(criterion='entropy', random_state=42)

In [30]:
y_pred = classifer.predict(x_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.to_numpy().reshape(len(y_test),1)), axis=1))

[[5 6]
 [5 5]
 [6 6]
 [5 5]
 [6 6]
 [5 5]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [7 7]
 [6 3]
 [6 5]
 [6 5]
 [5 6]
 [7 7]
 [6 5]
 [6 7]
 [8 8]
 [5 5]
 [5 5]
 [6 6]
 [6 5]
 [4 6]
 [6 6]
 [6 6]
 [6 7]
 [6 6]
 [6 5]
 [6 6]
 [3 5]
 [7 5]
 [6 6]
 [6 5]
 [6 6]
 [5 5]
 [6 7]
 [5 5]
 [6 4]
 [6 6]
 [7 5]
 [5 5]
 [6 7]
 [6 5]
 [5 5]
 [5 6]
 [6 7]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 7]
 [5 6]
 [6 6]
 [6 6]
 [5 5]
 [6 5]
 [5 5]
 [5 5]
 [5 7]
 [6 5]
 [5 6]
 [8 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [6 4]
 [5 6]
 [4 6]
 [6 6]
 [5 5]
 [6 8]
 [5 5]
 [5 6]
 [6 6]
 [5 5]
 [7 6]
 [5 5]
 [6 6]
 [6 6]
 [7 7]
 [5 5]
 [6 6]
 [6 7]
 [7 4]
 [8 7]
 [7 6]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [5 5]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [6 5]
 [7 7]
 [6 6]
 [7 7]
 [5 6]
 [7 5]
 [4 6]
 [6 5]
 [7 8]
 [5 5]
 [7 6]
 [5 5]
 [5 6]
 [7 7]
 [5 6]
 [6 6]
 [6 5]
 [6 6]
 [4 6]
 [6 6]
 [6 6]
 [6 6]
 [5 6]
 [5 6]
 [7 7]
 [6 6]
 [5 5]
 [4 5]
 [5 6]
 [6 5]
 [5 5]
 [6 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 5]
 [6 6]
 [7 7]
 [4 6]
 [7 8]

In [33]:
accuracy_score(y_test, y_pred)


0.584375

In [34]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 42)
classifier.fit(x_train, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=42)

In [35]:
y_pred = classifer.predict(x_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.to_numpy().reshape(len(y_test),1)), axis=1))

[[5 6]
 [5 5]
 [6 6]
 [5 5]
 [6 6]
 [5 5]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [7 7]
 [6 3]
 [6 5]
 [6 5]
 [5 6]
 [7 7]
 [6 5]
 [6 7]
 [8 8]
 [5 5]
 [5 5]
 [6 6]
 [6 5]
 [4 6]
 [6 6]
 [6 6]
 [6 7]
 [6 6]
 [6 5]
 [6 6]
 [3 5]
 [7 5]
 [6 6]
 [6 5]
 [6 6]
 [5 5]
 [6 7]
 [5 5]
 [6 4]
 [6 6]
 [7 5]
 [5 5]
 [6 7]
 [6 5]
 [5 5]
 [5 6]
 [6 7]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 7]
 [5 6]
 [6 6]
 [6 6]
 [5 5]
 [6 5]
 [5 5]
 [5 5]
 [5 7]
 [6 5]
 [5 6]
 [8 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [6 4]
 [5 6]
 [4 6]
 [6 6]
 [5 5]
 [6 8]
 [5 5]
 [5 6]
 [6 6]
 [5 5]
 [7 6]
 [5 5]
 [6 6]
 [6 6]
 [7 7]
 [5 5]
 [6 6]
 [6 7]
 [7 4]
 [8 7]
 [7 6]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [5 5]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [6 5]
 [7 7]
 [6 6]
 [7 7]
 [5 6]
 [7 5]
 [4 6]
 [6 5]
 [7 8]
 [5 5]
 [7 6]
 [5 5]
 [5 6]
 [7 7]
 [5 6]
 [6 6]
 [6 5]
 [6 6]
 [4 6]
 [6 6]
 [6 6]
 [6 6]
 [5 6]
 [5 6]
 [7 7]
 [6 6]
 [5 5]
 [4 5]
 [5 6]
 [6 5]
 [5 5]
 [6 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 5]
 [6 6]
 [7 7]
 [4 6]
 [7 8]

In [37]:
accuracy_score(y_test,y_pred)

0.584375

changing the no. of decision tree

In [38]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 40, criterion = 'entropy', random_state = 42)
classifier.fit(x_train, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=40, random_state=42)

In [39]:
y_pred = classifer.predict(x_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.to_numpy().reshape(len(y_test),1)), axis=1))

[[5 6]
 [5 5]
 [6 6]
 [5 5]
 [6 6]
 [5 5]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [7 7]
 [6 3]
 [6 5]
 [6 5]
 [5 6]
 [7 7]
 [6 5]
 [6 7]
 [8 8]
 [5 5]
 [5 5]
 [6 6]
 [6 5]
 [4 6]
 [6 6]
 [6 6]
 [6 7]
 [6 6]
 [6 5]
 [6 6]
 [3 5]
 [7 5]
 [6 6]
 [6 5]
 [6 6]
 [5 5]
 [6 7]
 [5 5]
 [6 4]
 [6 6]
 [7 5]
 [5 5]
 [6 7]
 [6 5]
 [5 5]
 [5 6]
 [6 7]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 7]
 [5 6]
 [6 6]
 [6 6]
 [5 5]
 [6 5]
 [5 5]
 [5 5]
 [5 7]
 [6 5]
 [5 6]
 [8 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [6 4]
 [5 6]
 [4 6]
 [6 6]
 [5 5]
 [6 8]
 [5 5]
 [5 6]
 [6 6]
 [5 5]
 [7 6]
 [5 5]
 [6 6]
 [6 6]
 [7 7]
 [5 5]
 [6 6]
 [6 7]
 [7 4]
 [8 7]
 [7 6]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [5 5]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [6 5]
 [7 7]
 [6 6]
 [7 7]
 [5 6]
 [7 5]
 [4 6]
 [6 5]
 [7 8]
 [5 5]
 [7 6]
 [5 5]
 [5 6]
 [7 7]
 [5 6]
 [6 6]
 [6 5]
 [6 6]
 [4 6]
 [6 6]
 [6 6]
 [6 6]
 [5 6]
 [5 6]
 [7 7]
 [6 6]
 [5 5]
 [4 5]
 [5 6]
 [6 5]
 [5 5]
 [6 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 5]
 [6 6]
 [7 7]
 [4 6]
 [7 8]

In [40]:
accuracy_score(y_test,y_pred)

0.584375

In [44]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 85, criterion = 'entropy', random_state = 42)
classifier.fit(x_train, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=85, random_state=42)

In [45]:
y_pred = classifer.predict(x_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.to_numpy().reshape(len(y_test),1)), axis=1))

[[5 6]
 [5 5]
 [6 6]
 [5 5]
 [6 6]
 [5 5]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [7 7]
 [6 3]
 [6 5]
 [6 5]
 [5 6]
 [7 7]
 [6 5]
 [6 7]
 [8 8]
 [5 5]
 [5 5]
 [6 6]
 [6 5]
 [4 6]
 [6 6]
 [6 6]
 [6 7]
 [6 6]
 [6 5]
 [6 6]
 [3 5]
 [7 5]
 [6 6]
 [6 5]
 [6 6]
 [5 5]
 [6 7]
 [5 5]
 [6 4]
 [6 6]
 [7 5]
 [5 5]
 [6 7]
 [6 5]
 [5 5]
 [5 6]
 [6 7]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 7]
 [5 6]
 [6 6]
 [6 6]
 [5 5]
 [6 5]
 [5 5]
 [5 5]
 [5 7]
 [6 5]
 [5 6]
 [8 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [5 5]
 [7 6]
 [6 4]
 [5 6]
 [4 6]
 [6 6]
 [5 5]
 [6 8]
 [5 5]
 [5 6]
 [6 6]
 [5 5]
 [7 6]
 [5 5]
 [6 6]
 [6 6]
 [7 7]
 [5 5]
 [6 6]
 [6 7]
 [7 4]
 [8 7]
 [7 6]
 [5 5]
 [5 5]
 [5 5]
 [6 6]
 [5 5]
 [5 6]
 [5 5]
 [5 6]
 [5 5]
 [5 5]
 [6 5]
 [7 7]
 [6 6]
 [7 7]
 [5 6]
 [7 5]
 [4 6]
 [6 5]
 [7 8]
 [5 5]
 [7 6]
 [5 5]
 [5 6]
 [7 7]
 [5 6]
 [6 6]
 [6 5]
 [6 6]
 [4 6]
 [6 6]
 [6 6]
 [6 6]
 [5 6]
 [5 6]
 [7 7]
 [6 6]
 [5 5]
 [4 5]
 [5 6]
 [6 5]
 [5 5]
 [6 5]
 [5 6]
 [5 5]
 [5 5]
 [5 5]
 [7 5]
 [6 6]
 [7 7]
 [4 6]
 [7 8]

In [46]:
accuracy_score(y_test,y_pred)

0.584375